# Summary Dashboard: Three NL Aces in 2026

**The definitive comparison.** This notebook assembles all findings into a single dashboard view.

**Core thesis:** Shohei Ohtani's pitching performance in 2026 is not as excellent as Cristopher Sanchez's or Jacob Misiorowski's.

In [ ]:
from analysis_utils import *
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

set_style()
dfs = load_all()
COLORS = {'Ohtani':'#1f77b4','Sanchez':'#ff7f0e','Misiorowski':'#2ca02c'}

summary = build_summary_df(dfs)
key_cols = ['Pitcher','Pitches','K%','BB%','K-BB%','Whiff%','AVG','OBP','SLG','OPS',
            'xwOBA','HardHit%','Barrels','GB','HR Allowed','RE per Pitch']
print("MASTER COMPARISON TABLE")
print("=" * 120)
print(summary[key_cols].to_string(index=False))

In [ ]:
# ---- Radar Chart ----
from math import pi
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
categories = ['K%','Whiff%','BB%\n(lower=better)','HardHit%\n(lower=better)',
              'xwOBA\n(lower=better)','RE/Pitch\n(lower=better)']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]
for name, df in dfs.items():
    kb = k_bb_stats(df); bb = batted_balls(df); pdm = plate_discipline(df)
    es = expected_stats(df); re = run_expectancy(df)
    vals = [kb.get('k_pct',0), pdm['whiff_pct'],
            15 - kb.get('bb_pct',0), 50 - bb.get('hard_hit_pct',0),
            0.500 - es.get('xwoba',0)*1000, 0.050 - abs(re.get('avg_re',0))*100]
    vals = [max(0, v) for v in vals]
    vals += vals[:1]
    ax.fill(angles, vals, alpha=0.1, color=COLORS[name])
    ax.plot(angles, vals, 'o-', linewidth=2, color=COLORS[name], label=name)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_title("Performance Radar (Outward = Better)", fontweight='bold', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('figures/radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Significance Summary ----
ohtani = dfs['Ohtani']; sanchez = dfs['Sanchez']; misio = dfs['Misiorowski']
def compare_kbb(p1, p2, metric):
    p1_pa = p1[p1['events'].notna() & (p1['events'] != '')]
    p2_pa = p2[p2['events'].notna() & (p2['events'] != '')]
    p1_e = len(p1_pa[p1_pa['events']==metric])
    p2_e = len(p2_pa[p2_pa['events']==metric])
    _, p, _, _ = chi2_contingency([[p1_e, len(p1_pa)-p1_e], [p2_e, len(p2_pa)-p2_e]])
    return p
p_bb = compare_kbb(ohtani, sanchez, 'walk')
p_k = compare_kbb(ohtani, misio, 'strikeout')
o_bip = ohtani[ohtani['description']=='hit_into_play']
s_bip = sanchez[sanchez['description']=='hit_into_play']
_, p_gb, _, _ = chi2_contingency([
    [len(o_bip[o_bip['bb_type']=='ground_ball']), len(o_bip)-len(o_bip[o_bip['bb_type']=='ground_ball'])],
    [len(s_bip[s_bip['bb_type']=='ground_ball']), len(s_bip)-len(s_bip[s_bip['bb_type']=='ground_ball'])]])
print("STATISTICAL SIGNIFICANCE\n" + "="*55)
for label, p in [('BB%: Ohtani vs Sanchez', p_bb),
                  ('K%: Ohtani vs Misiorowski', p_k),
                  ('GB%: Ohtani vs Sanchez', p_gb)]:
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    print(f"  {label:35s}  p={p:.4f}  {sig}")
print("\n*** p<0.001  ** p<0.01  * p<0.05  ns = not significant")

In [ ]:
# ---- Final Verdict ----
print("=" * 70)
print("FINAL COMPARISON VERDICT".center(70))
print("=" * 70)
print(f"{'Category':20s} {'Ohtani':12s} {'Sanchez':12s} {'Misiorowski':12s}")
print("-" * 56)
print(f"{'Control (BB%)':20s} {'7.6%':12s} {'4.8%':12s} {'6.4%':12s}   <- Sanchez")
print(f"{'Whiff%':20s} {'28.4%':12s} {'29.7%':12s} {'34.3%':12s}   <- Misiorowski")
print(f"{'K%':20s} {'27.9%':12s} {'27.6%':12s} {'39.6%':12s}   <- Misiorowski")
print(f"{'OPS Allowed':20s} {'.474':12s} {'.629':12s} {'.410':12s}   <- Misiorowski")
print(f"{'RE per pitch':20s} {'+0.0158':12s} {'+0.0080':12s} {'+0.0222':12s}   <- Sanchez")
print(f"{'HardHit%':20s} {'33.8%':12s} {'44.3%':12s} {'34.2%':12s}   <- Ohtani")
print(f"{'Barrels':20s} {'0':12s} {'8':12s} {'3':12s}   <- Ohtani")
print(f"{'Ground Balls':20s} {'111':12s} {'192':12s} {'101':12s}   <- Sanchez")
print()
print("CONCLUSION:".center(70))
print("Sanchez is the most efficient run preventer (lowest RE delta).")
print("Misiorowski is the most dominant in terms of raw stuff.")
print("Ohtani leads contact suppression, but trails in control,")
print("whiff rate, and run prevention.")
print("The data does NOT support Ohtani > Sanchez or Misiorowski.")